1. Structured Data - Transformation involves Standard SQL and DML operations
2. Semi-Structured Data (JSON, Parquet, XML)
Snowflake uses special data type called Variant to store semi structured data and no schema required
Transformation:
- Dot notation : Access nested fields directly
- Flatten: A table function used to explode arrays into individual rows
3. Unstructured Data (PDFs, Images, Audio, Logs)
Snowflake doesn't store the content of unstructured files in a table cell,
instead it stores metadata about the file and provides a directory table to manage them
Transformation:
- scoped url : generate temporary secure links to files for processing by external tools
- snowpark : since SQL can't read a PDF, you use Snowpark to run Python libraries(Pypdf and opencv) directly inside SNowflake
- User Defined function : write python UDF to extract text from a blob and return it as structured string.

# APPLYING SQL FOR QUERY OPTIMIZATION

1. The golden rule:  pruning : 
Snowflake stores data in micro-partitions. Your goal is to write SQL that allows Snowflake to ignore as many partiitons as possible, this is called pruning.

Main enemy : Functions on columns
slow : select * from table where year(date_col)=2024;
fast : select * from table where date_col>='2024-01-01' and date_col<'2025-01-01';

2. Reduce the Working Set:
Snowflake is columnar database. Every extra column you select adds to the processing cost.
avoid : select * from table , select only required columns
filter early : place you where condition as early as possible maybe CTEs

3. Join Optimization & Deduplication :
Joins are often the most expensive part of the query. SO prefer using Hash joinds, also see the possibility of join based on cluster keys and avoid cartesian product, i.e full outer join

While Deduplication often yield the same product, use group by instead of distinct
slow: select distinct user from tbl;
fast: select user from tbl group by 1;

4. Result Cache: Snowflake uses result cache if you use the same query without changing the underlying data and not use any non deterinistic function.

5. Sorting with order by:
Sorting is memory intensive operation
only sort at the very end: dont use order by inside CTEs, unless using a limit
use top N: if you need only top 10 rows, always use limit 10

# Window Function

are analytic functions that you can use for various calculations such as running totals, moving averages and rankings

e.g: lag, lead, any_value, count, list_agg, rank, dense_rank, nth_value